# Paradigm A: Perceptual Decision-Making — Training + Fixed-Point Analysis

This notebook trains a CTRNN on PerceptualDecisionMaking task and DelayComparison task,
then use **numerical fixed-point search** to uncover the decision geometry (line attractor).

The task data used in the tutorial is based on [NeuroGym](https://neurogym.github.io/neurogym/latest/), which is a curated collection of neuroscience tasks with a common interface to facilitate training of neural network models. The corresponding task description can be found in [this document](https://neurogym.github.io/neurogym/latest/api/tags/perceptual/)

> Requires: `pip install -e .'` or manually install NeuroGym first and test with `import neurogym`.

## 0. Setup

In [ ]:
import sys
sys.path.append('../src')
import pandas as pd
import torch, numpy as np, matplotlib.pyplot as plt
from neuralrnn import AutoConfig, AutoModel, Trainer, TrainingArguments
from neuralrnn import SupervisedObjective, load_dataset
# Fit PCA on all collected trial activity
from neuralrnn.analysis import fit_pca
from neuralrnn.analysis import find_fixed_points, linearize, dominant_direction

torch.manual_seed(0); np.random.seed(0)

## 1. Task data (neurogym)

In [ ]:
ds = load_dataset('perceptual_decision_making', batch_size=16, seq_len=100, dt=100)
print('input_dim =', ds.input_dim, '| n_actions =', ds.output_dim)

### Visualize sample trials

Plot a few trials to verify the task structure. The perceptual decision making requires integrating and comparing the left and right stimuli.
* Top row: input channels (fixation, stimulus-left, stimulus-right).
* Bottom row: target output (ground truth action).

In [ ]:
# Run a few trials and visualize input/output
env = ds.env
n_show = 3
colors = plt.cm.Set1(np.linspace(0, 1, n_show))

fig, axes = plt.subplots(ds.input_dim + 1, 1, figsize=(6, 4), sharex=True)
trial_labels = []
for i in range(n_show):
    env.new_trial()
    ob = env.ob  # (T, input_dim) -- time-first
    gt = env.gt  # (T,)
    t = np.arange(ob.shape[0]) * env.dt / 1000  # convert to seconds

    # Plot each input channel as a separate subplot
    for ch in range(ds.input_dim):
        axes[ch].plot(t, ob[:, ch], color=colors[i], alpha=0.8, lw=1.2)

    # Plot ground truth (target) at bottom
    axes[-1].plot(t, gt, color=colors[i], alpha=0.8, lw=1.2)

    coh = env.trial.get('coh', 0)
    gt_label = env.trial.get('ground_truth', '?')
    trial_labels.append(f'trial {i}: groundtruth={gt_label}, coh={coh}')

# Label input channel subplots
channel_names = ['Fixation', 'Stim-L', 'Stim-R']
for ch in range(ds.input_dim):
    axes[ch].set_ylabel(channel_names[ch] if ch < len(channel_names) else f'In {ch}')
    axes[ch].tick_params(labelsize=8)
axes[-1].set_ylabel('Target')
axes[-1].set_xlabel('Time (s)')
axes[-1].tick_params(labelsize=8)

fig.legend(trial_labels, bbox_to_anchor=(1, 0), fontsize=8, frameon=False, ncols=3)
fig.suptitle('Sample trials: inputs (top 3 rows) and target output (bottom)', fontsize=10)
plt.tight_layout()
plt.show()

## 2. CTRNN + supervised training

NeuralRNN provides a two-step pipeline to get the RNN.
* **Building the model**: NeuralRNN follows the **transformers-like `AutoConfig` / `AutoModel` pattern** (see API for detailed description). Specifically, we set parameters including model_type, input/latent/output dim, dt (time constant), and other model-specific parameters (optional) in `AutoConfig` to define the model configuration. Then we use AutoModel to load and init the model. 
    
    In this notebook, we use the **CTRNN** (Continuous-Time RNN), a standard model for cognitive task optimization. Its dynamics follow:

    $$z_{t+1} = (1-\alpha)z_t + \alpha \cdot f(W_{rec} z_t + W_{in} x_t + b)$$

    where $\alpha = dt/\tau$ controls the timescale and $f$ is the activation function.

* **Training the model**: 

In [ ]:
# Key step
cfg = AutoConfig.for_model('ctrnn', input_dim=ds.input_dim, latent_dim=32,
                           output_dim=ds.output_dim, dt=20, tau=100, 
                           # activation='sigmoid', # default is relu
                           # if True, nonlinearity is after blending: f((1-α)z + α·pre)
                           relu_after_blend=False)  
model = AutoModel.from_config(cfg)
objective = SupervisedObjective(task_type='classification')

history = Trainer(
            model, ds, objective, 
            TrainingArguments(max_steps=2000, learning_rate=1e-3, log_every=200, 
                          # device='cuda', # you can set device to 'cuda' in TrainingArguments if GPU is available
                          )
        ).train()

In [ ]:
# ---- Try to save and load a previously trained model ----
import os
SAVE_DIR = "./models/01/ctrnn"
os.makedirs(SAVE_DIR, exist_ok=True)
model.save_pretrained(SAVE_DIR)
print(f"\nModel saved to {SAVE_DIR}/")

model_loaded = AutoModel.from_pretrained(SAVE_DIR)
print(f"\nModel loaded from {SAVE_DIR}/")

# compare the weights of the original and loaded models
original_weights = model.h2h.weight.detach().cpu().numpy()
loaded_weights = model_loaded.h2h.weight.detach().cpu().numpy()
print(f"Are the weights equal? {np.allclose(original_weights, loaded_weights)}")

## 3. Collect activity + PCA

Run the trained model on many trials to collect per-trial neural activity,
ground truth, and coherence. Then fit PCA and produce a publication-quality
visualization colored by condition.

In [ ]:
# Collect per-trial activity, ground truth, coherence, and predicted actions
num_trial = 500
activity_dict = {}
trial_infos = {}
action_dict = {}

model.eval()
env = ds.env
with torch.no_grad():
    for i in range(num_trial):
        env.new_trial()
        ob, gt = env.ob, env.gt  # (T, input_dim), (T,)
        inputs_t = torch.from_numpy(ob[np.newaxis, :, :]).float()  # (1, T, input_dim) -- batch-first
        out = model(inputs_t)
        rnn_act = out.states    # (1, T, latent_dim)
        logits = out.outputs    # (1, T, output_dim)
        activity_dict[i] = rnn_act[0, :, :].numpy()  # (T, hidden_dim)
        trial_infos[i] = env.trial
        # argmax()-1: neurogym actions are 1-indexed (0=fix, 1=choice0, 2=choice1),
        # ground_truth is 0-indexed (0 or 1)
        action_dict[i] = logits[0, -1, :].numpy().argmax() - 1

# Build DataFrame and compute accuracy
df_trials = pd.DataFrame(trial_infos).T
df_trials['action'] = pd.Series(action_dict)
acc = (df_trials['action'] == df_trials['ground_truth']).mean()
print(f'Accuracy: {acc:.3f}')
print(f'Trial info example: {trial_infos[0]}')
print(f'Unique coherences: {sorted(df_trials["coh"].unique())}')

# Concatenate all activity for PCA fitting
activity_all = np.concatenate([activity_dict[i] for i in range(num_trial)], axis=0)
print(f'Activity shape for PCA: {activity_all.shape}')

In [ ]:
pca = fit_pca(activity_all, n_components=2)
print('Explained variance ratio:', np.round(pca.explained_variance_ratio, 3))

# --- Publication-quality PCA visualization ---
# Color scheme: green for ground_truth=0, purple for ground_truth=1
# Intensity modulated by coherence level
colors_rgb = np.array([[27, 158, 119], [117, 112, 179]]) / 255.  # green, purple

cohs = sorted(df_trials['coh'].unique())
cohs_pos = [c for c in cohs if c > 0]
color_intensity = [0.4, 0.7, 1.0, 1.3][:len(cohs_pos)]

fig = plt.figure(figsize=(4, 4))
ax = fig.add_axes([0.2, 0.2, 0.6, 0.6])

for ground_truth in [0, 1]:
    if ground_truth == 0:
        cohs_ = cohs_pos[::-1]
        color_intensity_ = color_intensity[::-1]
    else:
        cohs_ = cohs_pos
        color_intensity_ = color_intensity
    for i_coh, coh in enumerate(cohs_):
        trials_idx = df_trials[(df_trials['coh'] == coh) &
                               (df_trials['ground_truth'] == ground_truth)].index
        if len(trials_idx) == 0:
            continue
        activity_avg = np.mean([activity_dict[i] for i in trials_idx], axis=0)
        activity_pc = pca.transform(activity_avg)
        color = colors_rgb[ground_truth] * color_intensity_[i_coh]
        signed_coh = coh * (2 * ground_truth - 1)
        ax.plot(activity_pc[:, 0], activity_pc[:, 1], 'o-',
                color=color, ms=3, markeredgecolor='none', lw=1,
                label=f'{signed_coh:+.1f}')

ax.legend(title='Signed Coh', loc='upper left', bbox_to_anchor=(1.0, 1.0),
          frameon=False, fontsize=7)
ax.set_xlabel('PC 1', fontsize=7)
ax.set_ylabel('PC 2', fontsize=7)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(labelsize=8)
plt.show()

## 4. Numerical fixed point search + line attractor

Search for approximate fixed points under the **0-coherence stimulus condition**
(fixation=1, left-stim=0.5, right-stim=0.5), then project onto the PCA plane
and draw the dominant eigenvector direction (line attractor).

In [ ]:
# 0-coherence stimulus input: fixation=1, left=0.5, right=0.5
# This matches the reference implementation (nn-brain tutorial)
task_input = torch.tensor([1, 0.5, 0.5], dtype=torch.float32)

# speed_tol controls the ||F(z)-z|| threshold for accepting converged candidates.
# Default 0.1 works well for typical 10k-step convergence (MSE ~1e-5 → speed ~0.01).
fps = find_fixed_points(model, backend='numeric', task_input=task_input,
                        n_candidates=128, n_iters=10000, speed_tol=0.5)
print(f'Found {len(fps)} fixed points')

# --- Visualize fixed points in PCA space ---
coords = pca.transform(fps.coords()) if len(fps) else np.empty((0, 2))

fig = plt.figure(figsize=(5, 5))
ax = fig.add_axes([0.15, 0.15, 0.75, 0.75])
colors_rgb = np.array([[27, 158, 119], [117, 112, 179]]) / 255.  # green, purple
# Overlay per-trial trajectories colored by ground truth
for i in range(min(100, num_trial)):
    activity_pc = pca.transform(activity_dict[i])
    color = colors_rgb[0] if trial_infos[i]['ground_truth'] == 0 else colors_rgb[1]
    ax.plot(activity_pc[:, 0], activity_pc[:, 1], '.', color=color, alpha=0.8, ms=2)

if len(fps):
    ax.plot(coords[:, 0], coords[:, 1], 'kx', ms=5, mew=0.5, label='Fixed points')
    # Dominant eigenvector direction at first fixed point
    lin = linearize(model, fps.points[0].z, task_input=task_input)
    d = dominant_direction(lin)
    seg = pca.transform(np.array([fps.points[0].z + 2 * d,
                                   fps.points[0].z - 2 * d]))
    ax.plot(seg[:, 0], seg[:, 1], 'r-', lw=2, label='Dominant eigenvector')

ax.set_xlabel('PC 1')
ax.set_ylabel('PC 2')
ax.legend(fontsize=8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.show()

---

**Summary**: The model implements `recurrence`/`readout`; fixed points, linearization, and PCA
all come from the generic `analysis` module, shared with Paradigm B tutorials.

**Key differences from a naive implementation** (that may cause no stable fixed points):
1. `task_input` must be the **0-coherence stimulus** `[1, 0.5, 0.5]`, NOT zeros.
   Zero input drives all activity to zero trivially — no interesting fixed points exist there.
2. Use >=10000 optimization steps for convergence (loss should reach ~0).
3. Initialize candidates in `[0, 3)` (positive) to match ReLU activity range.
4. `speed_tol` (the `||F(z)−z||` acceptance threshold) must be set appropriately for the
   convergence level. After 10k steps the MSE is typically ~1e-5, giving speed ~0.01–0.04.
   The default `speed_tol=1e-5` was far too strict; `0.1` works well in practice.
   If you still get 0 fixed points, try increasing `n_iters` or relaxing `speed_tol`.

---

# Part B: Parametric Working Memory — Training + Line Attractor Analysis

Reproduce the second nn-brain tutorial: train a CTRNN on neurogym's **DelayComparison** (parametric working memory) task, then use numerical fixed-point search and Jacobian analysis to uncover the **line attractor** that supports working memory.

> This section continues from Part A. The same `analysis` module APIs are used.

## B1. Task: DelayComparison (Parametric Working Memory)

In this task, the network must compare two stimuli (v1, v2) separated by a variable delay, and report which was larger. The key feature: the delay period varies from 200ms to 3200ms, requiring the network to maintain a working memory of the first stimulus.

In [ ]:
# Load the DelayComparison task via the registered dataset
timing = {'delay': ('choice', [200, 400, 800, 1600, 3200]),
          'response': ('constant', 500)}

ds_pw = load_dataset('delay_comparison', timing=timing,
                     batch_size=16, seq_len=100, dt=100)
print('input_dim =', ds_pw.input_dim, '| n_actions =', ds_pw.output_dim)

# Visualize sample trials
env_pw = ds_pw.env
n_show = 2
fig, axes = plt.subplots(ds_pw.input_dim + 1, 1, figsize=(8, 4), sharex=True)
colors = plt.cm.Set1(np.linspace(0, 1, n_show))
trial_labels = []
for i in range(n_show):
    env_pw.new_trial()
    ob = env_pw.ob
    gt = env_pw.gt
    t = np.arange(ob.shape[0]) * env_pw.dt / 1000
    for ch in range(ds_pw.input_dim):
        axes[ch].plot(t, ob[:, ch], color=colors[i], alpha=0.8, lw=1.2)
    axes[-1].plot(t, gt, color=colors[i], alpha=0.8, lw=1.2)
    trial = env_pw.trial
    trial_labels.append(f'trial {i}: v1={trial.get("v1", "?")}, v2={trial.get("v2", "?")}')

for ch in range(ds_pw.input_dim):
    axes[ch].set_ylabel(f'Stim {ch}')
axes[-1].set_ylabel('Target')
axes[-1].set_xlabel('Time (s)')
fig.legend(trial_labels, loc='upper right', fontsize=8)
fig.suptitle('Parametric Working Memory: DelayComparison task', fontsize=10)
plt.tight_layout()
plt.show()

## B2. Train CTRNN on DelayComparison

In [ ]:
# Train a new CTRNN on the DelayComparison task
cfg_pw = AutoConfig.for_model('ctrnn', input_dim=ds_pw.input_dim, latent_dim=64,
                               output_dim=ds_pw.output_dim, dt=100, tau=100)
model_pw = AutoModel.from_config(cfg_pw)
objective_pw = SupervisedObjective(task_type='classification')
history_pw = Trainer(model_pw, ds_pw, objective_pw,
                     TrainingArguments(max_steps=2000, learning_rate=1e-3, log_every=200)).train()

## B3. Collect activity + PCA visualization

Run the trained model on many trials with a **fixed long delay (3000ms)** for analysis. This makes the delay period long enough to clearly see the dynamics.

In [ ]:
# Create environment with fixed long delay for analysis
timing_analysis = {'delay': ('constant', 2000), 'response': ('constant', 500)}
ds_analysis = load_dataset('delay_comparison', timing=timing_analysis,
                           batch_size=16, seq_len=100, dt=100)
env_analysis = ds_analysis.env

# Collect activity during delay period
num_trial_pw = 100
activity_dict_pw = {}
trial_infos_pw = {}

model_pw.eval()
with torch.no_grad():
    for i in range(num_trial_pw):
        env_analysis.new_trial()
        ob, gt = env_analysis.ob, env_analysis.gt
        inputs_t = torch.from_numpy(ob[np.newaxis, :, :]).float()
        out = model_pw(inputs_t)
        rnn_act = out.states[0, :, :].numpy()
        # Extract activity during delay period only
        activity_dict_pw[i] = rnn_act[env_analysis.start_ind['delay']:env_analysis.end_ind['delay']]
        trial_infos_pw[i] = env_analysis.trial.copy()

# Concatenate delay-period activity for PCA
activity_pw = np.concatenate([activity_dict_pw[i] for i in range(num_trial_pw)], axis=0)
print('Shape of delay-period neural activity:', activity_pw.shape)

# Print sample trial info
for i in range(5):
    print(f'Trial {i}:', trial_infos_pw[i])

In [ ]:
# Fit PCA on delay-period activity
pca_pw = fit_pca(activity_pw, n_components=2)
print('Explained variance ratio:', np.round(pca_pw.explained_variance_ratio, 3))

# Visualize individual trials colored by ground truth
fig, (ax1, ax2) = plt.subplots(1, 2, sharey=True, sharex=True, figsize=(4, 2))
for i in range(num_trial_pw):
    trial = trial_infos_pw[i]
    activity_pc = pca_pw.transform(activity_dict_pw[i])
    color = 'red' if trial['ground_truth'] == 1 else 'blue'
    ax1.plot(activity_pc[:, 0], activity_pc[:, 1], 'o-', color=color,
             markersize=1, lw=0.5, alpha=0.5)
    if i < 1:
        ax2.plot(activity_pc[:, 0], activity_pc[:, 1], 'o-', color=color)

ax1.set_xlabel('PC 1')
ax1.set_ylabel('PC 2')
ax1.set_title('All trials')
ax2.set_title('Single trial')
plt.tight_layout()
plt.show()

## B4. Fixed-point search for the line attractor

Search for approximate fixed points under the **0-coherence stimulus** (v1=v2, so the network should maintain equal memory of both stimuli). This reveals the line attractor along which the working memory is maintained.

In [ ]:
# Fixed point search with 0-coherence input (v1=v2)
# Input format: [stimulus1, stimulus2] - both equal for 0-coherence
task_input_pw = torch.tensor([1, 0], dtype=torch.float32)

fps_pw = find_fixed_points(model_pw, backend='numeric', task_input=task_input_pw,
                            n_candidates=128, n_iters=10000, speed_tol=0.5)
print(f'Found {len(fps_pw)} fixed points')

# Visualize fixed points in PCA space
coords_pw = pca_pw.transform(fps_pw.coords()) if len(fps_pw) else np.empty((0, 2))

fig = plt.figure(figsize=(3, 3))
ax = fig.add_axes([0.15, 0.15, 0.75, 0.75])
for i in range(min(20, num_trial_pw)):
    activity_pc = pca_pw.transform(activity_dict_pw[i])
    trial = trial_infos_pw[i]
    color = 'red' if trial['ground_truth'] == 0 else 'blue'
    ax.plot(activity_pc[:, 0], activity_pc[:, 1], 'o-', color=color, alpha=0.1,
            markersize=1, lw=0.5)

if len(fps_pw):
    ax.plot(coords_pw[:, 0], coords_pw[:, 1], 'x', ms=5, color='black', label='Fixed points')

ax.set_xlabel('PC 1')
ax.set_ylabel('PC 2')
ax.set_title('Fixed points in delay-period PCA space')
ax.legend(fontsize=8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.show()

## B5. Jacobian analysis and line attractor

Compute the Jacobian at a fixed point near the center of the attractor manifold, then find the dominant eigenvector direction. This reveals the **line attractor** along which evidence is integrated.

In [ ]:
# Compute Jacobian at a central fixed point
if len(fps_pw) > 0:
    # Choose fixed point closest to center (by sorting PC1)
    fp_coords = fps_pw.coords()
    i_fp = np.argsort(fp_coords[:, 0])[len(fp_coords) // 2]
    
    # Linearize around this fixed point
    lin = linearize(model_pw, fps_pw.points[i_fp].z, task_input=task_input_pw)
    d = dominant_direction(lin)
    
    # Project dominant direction onto PCA space
    fp_z = fps_pw.points[i_fp].z
    end_pts = np.array([fp_z + 2 * d, fp_z - 2 * d])
    end_pts_pc = pca_pw.transform(end_pts)
    
    # Plot
    fig = plt.figure(figsize=(3, 3))
    ax = fig.add_axes([0.15, 0.15, 0.75, 0.75])
    
    for i in range(min(20, num_trial_pw)):
        activity_pc = pca_pw.transform(activity_dict_pw[i])
        trial = trial_infos_pw[i]
        color = 'red' if trial['ground_truth'] == 0 else 'blue'
        ax.plot(activity_pc[:, 0], activity_pc[:, 1], 'o-', color=color, alpha=0.1,
                markersize=1, lw=0.5)
    
    # Plot fixed points
    ax.plot(coords_pw[:, 0], coords_pw[:, 1], 'x', ms=5, color='black', alpha=0.3)
    ax.plot(coords_pw[i_fp, 0], coords_pw[i_fp, 1], 'x', ms=8, color='red', mew=2,
            label='Selected FP')
    
    # Plot line attractor direction
    ax.plot(end_pts_pc[:, 0], end_pts_pc[:, 1], '-', color='red', lw=2,
            label='Dominant eigenvector')
    
    ax.set_xlabel('PC 1')
    ax.set_ylabel('PC 2')
    ax.set_title('Line attractor revealed by Jacobian analysis')
    ax.legend(fontsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.show()
    
    # Print eigenvalue information
    eigvals = lin.eigenvalues
    print(f'Eigenvalues at selected FP (top 5 by magnitude):')
    sorted_idx = np.argsort(-np.abs(eigvals))
    for idx in sorted_idx[:5]:
        print(f'  λ = {eigvals[idx]:.4f}, |λ| = {np.abs(eigvals[idx]):.4f}')
else:
    print('No fixed points found. Try relaxing speed_tol or increasing n_iters.')

## B6. Publication-quality visualization

Create a publication-quality figure showing the neural dynamics organized by stimulus value (v1), with and without the line attractor overlay.

In [ ]:
# Publication-quality figure: dynamics colored by stimulus value
import pandas as pd
import matplotlib as mpl

# Build DataFrame from trial info
df_pw = pd.DataFrame(trial_infos_pw).T

# Two-panel figure: without and with fixed points
fig, axes = plt.subplots(1, 2, sharex=True, sharey=True, figsize=(4, 2))
colors_rgb = np.array([[27, 158, 119], [117, 112, 179], [217, 95, 2]]) / 255.

for panel_idx in range(2):
    ax = axes[panel_idx]
    plot_fp = (panel_idx == 1)
    
    # Group by v1 stimulus value
    values = np.unique(df_pw['v1'])
    cmap = mpl.cm.get_cmap('winter')
    alpha = 0.2 if plot_fp else 1.0
    
    for i, val in enumerate(values):
        trials = df_pw[df_pw['v1'] == val].index
        if len(trials) == 0:
            continue
        activity = np.mean([activity_dict_pw[i] for i in trials], axis=0)
        activity_pc = pca_pw.transform(activity)
        label = f'{val:.1f}'
        color = cmap(i / len(values))
        ax.plot(activity_pc[:, 0], activity_pc[:, 1], 'o-',
                color=color, ms=3, markeredgecolor='none',
                lw=1, label=label, alpha=alpha)
        ax.plot(activity_pc[0, 0], activity_pc[0, 1], 'o-', alpha=alpha,
                marker='^', color=color, ms=5)
    
    if plot_fp and len(fps_pw) > 0:
        # Overlay fixed points and line attractor
        color = colors_rgb[2]
        ax.plot(coords_pw[:, 0], coords_pw[:, 1], 'x', ms=3, color=color, alpha=0.3)
        ax.plot(coords_pw[i_fp, 0], coords_pw[i_fp, 1], 'x', ms=5, color=color, lw=1)
        ax.plot(end_pts_pc[:, 0], end_pts_pc[:, 1], color=color)
    else:
        ax.legend(title='Stimulus', loc='upper left', bbox_to_anchor=(1.0, 1.0), frameon=False,
                  fontsize=6)
    
    ax.set_xlabel('PC 1', fontsize=7)
    ax.set_ylabel('PC 2', fontsize=7)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.locator_params(nbins=2)
plt.show()

## B7. Eigenvalue distribution

Visualize the eigenvalues of the Jacobian in the complex plane to assess stability properties.

In [ ]:
# Plot eigenvalue distribution in complex plane
if len(fps_pw) > 0:
    eigvals = lin.eigenvalues
    
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.scatter(np.real(eigvals), np.imag(eigvals), s=30, alpha=0.7)
    ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    
    # Draw unit circle for reference
    theta = np.linspace(0, 2 * np.pi, 100)
    ax.plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.3, lw=0.5)
    
    ax.set_xlabel('Real', fontsize=10)
    ax.set_ylabel('Imaginary', fontsize=10)
    ax.set_title('Eigenvalue distribution', fontsize=10)
    ax.set_aspect('equal')
    plt.tight_layout()
    plt.show()
    
    # Count stable eigenvalues (inside unit circle)
    n_stable = np.sum(np.abs(eigvals) < 1)
    print(f'Stable eigenvalues (|λ|<1): {n_stable}/{len(eigvals)}')
else:
    print('No fixed points found - cannot compute eigenvalues.')

---

**Summary (Part B)**: The DelayComparison task requires the network to maintain a working memory of the first stimulus during the delay period. The fixed-point analysis reveals a **line attractor** — a manifold of approximately stable fixed points along which the memory is maintained. The dominant eigenvector of the Jacobian at these fixed points aligns with this line attractor, confirming the dynamical mechanism underlying parametric working memory.

**Key points**:
1. The delay period must be long enough (≥2000ms) to clearly see the attractor dynamics
2. Use 0-coherence input (v1=v2) for fixed-point search to find the center of the attractor manifold
3. The line attractor corresponds to PC1 in the PCA space — different stimulus values map to different positions along this line